In [40]:
from pathlib import Path
import pandas as pd
from credit_card_fraud_detection.paths import ROOT, RAW
from datetime import date
from pydantic import BaseModel, Field, ValidationError, ConfigDict

In [4]:
train = RAW / "train.csv"
test = RAW / "test.csv"

In [7]:
train_df = pd.read_csv(train)
train_df.columns

Index(['id', 'amount_usd', 'hours_since_last_txn', 'txn_count_last_24h',
       'distance_from_home_km', 'card_age_months', 'customer_age',
       'account_balance_usd', 'cvv_retry_count', 'velocity_score',
       'time_of_day_hour', 'day_of_week', 'merchant_risk_score',
       'prior_disputes', 'merchant_category', 'card_type', 'auth_method',
       'channel', 'device_type', 'is_foreign_transaction', 'is_new_merchant',
       'used_vpn', 'ip_country_mismatch', 'billing_shipping_mismatch',
       'is_ai_generated_scam_attempt', 'is_fraud'],
      dtype='object')

In [10]:
train_df.head()

,id,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,...,auth_method,channel,device_type,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt,is_fraud
0,0,61.16,9.77,1,17.78,48,56,1269.35,0,5.5,...,OTP,POS,iPhone,False,False,False,False,False,False,0.0
1,1,24.83,1.54,3,13.06,38,28,3233.00,0,33.4,...,PIN,In-App,Android Phone,False,False,False,False,False,False,0.0
2,2,6.36,1.20,4,8.17,50,55,1046.88,0,35.8,...,OTP,Online,Tablet,False,False,True,False,False,False,0.0
3,3,37.44,5.13,4,8.67,48,48,2217.67,0,20.7,...,3D Secure,POS,Android Phone,False,False,False,False,False,False,0.0
4,4,653.85,0.80,4,11.69,37,52,3291.06,0,34.8,...,3D Secure,Online,Windows PC,False,False,False,False,False,False,0.0


In [ ]:
# model config allow what it does it, allows columns thats not defined to be included
class CreditCardFraud(BaseModel):
    model_config = ConfigDict(extra="allow")
    id: int
    customer_age: int = Field(gt=17)

In [34]:

# for catching errors
row = {"id": 1, "customer_age": 17}
try:
    CreditCardFraud.model_validate(row)
except ValidationError as e:
    for err in e.errors():
        print(err["loc"], err["type"], "|", err["msg"], "|", repr(err["input"])[:30])
#print(democreditcard.id, type(democreditcard.customer_age).__name__)

# convert valid one into normal dataframe
c = CreditCardFraud.model_validate({"id": 1, "customer_age": 18})
print(c.model_dump())
print(c.model_dump(mode="json"))

('customer_age',) greater_than | Input should be greater than 17 | 17
{'id': 1, 'customer_age': 18}
{'id': 1, 'customer_age': 18}


In [46]:
good, bad = [], []

for i, row in enumerate(train_df.to_dict("records")):
    try:
        good.append(CreditCardFraud.model_validate(row))
    except Exception as e:
        bad.append((i, e.errors()))
print(len(good), "valid;", len(bad), "rejected")

699960 valid; 0 rejected


In [50]:
# make valid data into seperate train df data
valid_train_df = pd.DataFrame([m.model_dump() for m in good])
valid_train_df.head()

,id,customer_age,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,account_balance_usd,cvv_retry_count,velocity_score,...,auth_method,channel,device_type,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt,is_fraud
0,0,56,61.16,9.77,1,17.78,48,1269.35,0,5.5,...,OTP,POS,iPhone,False,False,False,False,False,False,0.0
1,1,28,24.83,1.54,3,13.06,38,3233.00,0,33.4,...,PIN,In-App,Android Phone,False,False,False,False,False,False,0.0
2,2,55,6.36,1.20,4,8.17,50,1046.88,0,35.8,...,OTP,Online,Tablet,False,False,True,False,False,False,0.0
3,3,48,37.44,5.13,4,8.67,48,2217.67,0,20.7,...,3D Secure,POS,Android Phone,False,False,False,False,False,False,0.0
4,4,52,653.85,0.80,4,11.69,37,3291.06,0,34.8,...,3D Secure,Online,Windows PC,False,False,False,False,False,False,0.0


In [ ]:
# since our whole dataset is valid, no need to save this